# 03 - Modelling walkthrough: from baselines to a trusted number

**Purpose.** This notebook tells the modelling story in order - *what we compared, how we tuned,
how we calibrated, and what survived contact with the test set* - so a reader can follow the
reasoning without reading the code. All numbers are read from the result files the pipeline
generated (`validation_results.json`, `results.json`); nothing is typed by hand.

**The one idea to keep in mind:** a model score is only meaningful relative to (a) a baseline and
(b) data the model never influenced. Every section below is about one of those two things.

**How to read:** each section is *question -> evidence -> inference*; inferences are computed so
they stay true if the pipeline is re-run.

In [1]:
import warnings

warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from ontime.reporting import walkthrough as w
from ontime.analysis.eda import set_style

set_style()
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
pct = lambda v: f"{v:.1%}"
res, val = w.results(), w.validation()
prim = res["primary_model"]

## 1. How the data was cut (all splits are by time)
**Why this matters:** the late rate is not constant over time, so *where* you cut decides what a
metric means. Tuning, calibrating and testing each get their own, later slice.

In [2]:
splits = w.split_table()
display(splits.set_index("split"))
roles = {
    "train": "fit models",
    "val_tune": "tune hyperparameters, early-stop",
    "calib_fit": "fit calibrators",
    "calib_eval": "check calibrators",
    "val_calib": "= calib_fit + calib_eval (honest validation)",
    "test": "judged once, at the end",
}
lo, hi = (
    splits.loc[splits["split"].isin(["train", "val_tune", "val_calib", "test"])]
    .sort_values("late_rate")
    .iloc[[0, -1]]["late_rate"]
)
Markdown(
    f"""
**Inference.** The late rate across these slices ranges from **{pct(lo)}** to **{pct(hi)}**.
A model tuned on a high-late slice and judged on a low-late slice faces a different world, so the
design keeps tuning (`val_tune`) and later-in-time honest evaluation (`val_calib`) apart, and the
test set is later still. Roles: """
    + "; ".join(f"`{k}` {v}" for k, v in roles.items())
    + "."
)

,from,to,orders,late_rate
split,,,,
train,2016-09-15,2018-01-31,50603,0.056
val_tune,2018-02-01,2018-03-15,10057,0.177
val_calib,2018-03-16,2018-04-30,10184,0.076
calib_fit,2018-03-16,2018-04-07,5141,0.114
calib_eval,2018-04-08,2018-04-30,5043,0.037
test,2018-05-01,2018-08-18,24093,0.045



**Inference.** The late rate across these slices ranges from **4.5%** to **17.7%**.
A model tuned on a high-late slice and judged on a low-late slice faces a different world, so the
design keeps tuning (`val_tune`) and later-in-time honest evaluation (`val_calib`) apart, and the
test set is later still. Roles: `train` fit models; `val_tune` tune hyperparameters, early-stop; `calib_fit` fit calibrators; `calib_eval` check calibrators; `val_calib` = calib_fit + calib_eval (honest validation); `test` judged once, at the end.

## 2. Baselines: what does "better than nothing" mean?
Two baselines set the floor: **guess the training late rate** for every order, and a
**hand-written rule** an analyst might write (threshold chosen on train only).

In [3]:
rule = res["baselines"]["rule"]
Markdown(f"""
- The rule found on the training period: **{rule["description"]}**.
- On the **test** period its PR-AUC is **{rule["metrics"]["pr_auc"]["value"]:.3f}** against a
  base rate of **{res["population"]["late_rate"]:.3f}**; its lift at 10% capacity is
  **{next(t for t in rule["top_k"] if abs(t["k"] - 0.1) < 1e-9)["lift"]:.2f}**.

**Inference.** A rule that looked useful on the training period is no better than guessing on the
test period. This is the cleanest illustration of why a *later* hold-out is the only honest test.
""")


- The rule found on the training period: **seller and customer in different states AND promised_days < 32.1**.
- On the **test** period its PR-AUC is **0.048** against a
  base rate of **0.045**; its lift at 10% capacity is
  **0.94**.

**Inference.** A rule that looked useful on the training period is no better than guessing on the
test period. This is the cleanest illustration of why a *later* hold-out is the only honest test.


## 3. Four model families and what each is for
| Model | Role | Interpretability |
|---|---|---|
| Logistic regression | transparent linear benchmark | coefficients -> odds ratios |
| EBM (Explainable Boosting Machine) | glass-box additive model | plot each feature's curve |
| LightGBM | strong gradient-boosted trees | post-hoc SHAP |
| Random forest | sanity comparison, fixed settings | post-hoc |

All share the same feature table. Linear models get imputation, missing-indicators, log-transform of
skewed quantities, scaling and one-hot encoding (fitted on train only); tree models consume raw
values and native categoricals.

## 4. Tuning: what did Optuna choose, and did class weighting help?
**Why we ask:** with a rare class, "re-weight the minority" is the textbook reflex. Here it is
a hyperparameter the validation data gets to vote on.

In [4]:
display(w.tuned_params().set_index("model"))
cw = w.class_weight_table()
display(cw.set_index(["model", "class weighting"]))
lr = cw[cw["model"] == "Logistic regression"].set_index("class weighting")["best PR-AUC (val_tune)"]
lg = cw[cw["model"] == "LightGBM"].set_index("class weighting")
Markdown(f"""
**Inference.**
- Logistic regression: weighting was **{"worse" if lr["balanced"] < lr["none"] else "better"}**
  (best {lr["none"]:.3f} unweighted vs {lr["balanced"]:.3f} balanced).
- LightGBM: the search favoured **{lg["best PR-AUC (val_tune)"].idxmax()}** weighting
  ({lg["best PR-AUC (val_tune)"].max():.3f}) but the optimizer concentrated its trials there
  ({", ".join(f"{k}: {int(v)} trials" for k, v in lg["trials"].items())}), so the comparison is weak
  evidence either way.
- Weighting changes the *scale* of scores, which is why calibration (Section 6) is needed afterwards.
""")

,chosen hyperparameters
model,
Logistic regression,"C=50.4, class_weight=none"
Random forest,"n_estimators=300, min_samples_leaf=50"
LightGBM,"num_leaves=19, learning_rate=0.0216, min_child..."
EBM,"max_bins=128, interactions=10, learning_rate=0..."


best PR-AUC (val_tune)  mean PR-AUC  \
model               class weighting                                        
Logistic regression none                              0.372        0.371   
                    balanced                          0.359        0.354   
LightGBM            none                              0.374        0.363   
                    sqrt                              0.379        0.366   
                    balanced                          0.393        0.381   

                                     trials  
model               class weighting          
Logistic regression none                 18  
                    balanced              7  
LightGBM            none                  3  
                    sqrt                  5  
                    balanced             42


**Inference.**
- Logistic regression: weighting was **worse**
  (best 0.372 unweighted vs 0.359 balanced).
- LightGBM: the search favoured **balanced** weighting
  (0.393) but the optimizer concentrated its trials there
  (none: 3 trials, sqrt: 5 trials, balanced: 42 trials), so the comparison is weak
  evidence either way.
- Weighting changes the *scale* of scores, which is why calibration (Section 6) is needed afterwards.


## 5. Tuned-on vs honest: how much of the score is real?
**Why we ask:** a score on the slice you tuned on is optimistic. The later `val_calib` slice was
never used by tuning, so the gap between the two columns measures how much was tuning optimism or
regime-specific structure.

In [5]:
tvh, base = w.tuned_vs_honest()
display(tvh.set_index("model"))
fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(tvh))
wd = 0.38
ax.bar(x - wd / 2, tvh["val_tune (tuned on)"], wd, label="val_tune (tuned on)", color="#4C78A8")
ax.bar(x + wd / 2, tvh["val_calib (honest)"], wd, label="val_calib (honest)", color="#EB6834")
ax.set_xticks(x, tvh["model"], rotation=20, ha="right")
ax.set_ylabel("PR-AUC")
ax.set_title("PR-AUC on the slice tuned on vs a later untouched slice")
for k, c in zip(base, ["#4C78A8", "#EB6834"], strict=True):
    ax.axhline(base[k], color=c, ls=":", lw=1)
ax.legend()
plt.tight_layout()
plt.show()
t = tvh.set_index("model")
ranked_tune = (
    t["val_tune (tuned on)"].drop(["Guess the training rate", "Hand-written rule"]).idxmax()
)
ranked_hon = t["val_calib (honest)"].drop(["Guess the training rate", "Hand-written rule"]).idxmax()
Markdown(f"""
**Inference.** The best model on the tuned-on slice is **{ranked_tune}**; on the later untouched
slice it is **{ranked_hon}**. Dotted lines are each slice's base rate (random ranking).
When the ranking of models changes between slices, the data is telling us the relationships are
not stable over time - so we **fixed the primary model by the honest column before opening the test
set**, rather than choosing the one that looked best where it was tuned.
""")

,train,val_tune (tuned on),val_calib (honest)
model,,,
Logistic regression,0.172,0.372,0.174
Random forest,0.350,0.360,0.126
LightGBM,0.318,0.393,0.137
EBM,0.203,0.395,0.122
Guess the training rate,0.056,0.177,0.076
Hand-written rule,0.066,0.230,0.102



**Inference.** The best model on the tuned-on slice is **EBM**; on the later untouched
slice it is **Logistic regression**. Dotted lines are each slice's base rate (random ranking).
When the ranking of models changes between slices, the data is telling us the relationships are
not stable over time - so we **fixed the primary model by the honest column before opening the test
set**, rather than choosing the one that looked best where it was tuned.


## 6. Calibration: making the number mean what it says
**Why we ask:** the app shows a probability to humans. Three options are compared by Brier score on
data *later* than the data the calibrator was fitted on: do nothing, Platt (sigmoid), isotonic.

In [6]:
cal = w.calibration_table()
display(cal.set_index("model"))
best_gain = (
    cal.assign(gain=cal["Brier none"] - cal[["Brier sigmoid", "Brier isotonic"]].min(axis=1))
    .sort_values("gain")
    .iloc[-1]
)
key = {v: k for k, v in w.LABEL.items()}[best_gain["model"]]
cw_opt = val["models"][key]["best_params"].get("class_weight", "none")
sp = splits.set_index("split")["late_rate"]
chosen = cal.set_index("model")["chosen"]
Markdown(f"""
**Inference.** The calibrator that helped most is for **{best_gain["model"]}**
(Brier {best_gain["Brier none"]:.3f} uncalibrated -> {min(best_gain["Brier sigmoid"], best_gain["Brier isotonic"]):.3f}).
That model's class-weighting option was **{cw_opt}**: {"its raw scores were on the wrong scale on purpose, so calibration rescued them" if cw_opt != "none" else "its raw scores were not re-weighted, so the gain is about shape rather than scale"}.
Choices by model: {", ".join(f"{m}: {c}" for m, c in chosen.items())}. Isotonic was
{"never" if "isotonic" not in set(chosen) else "sometimes"} selected: flexibility did not pay for
itself. Caveat from the data: the late rate was **{pct(sp["calib_fit"])}** in the window the calibrators
were fitted on and **{pct(sp["calib_eval"])}** in the window they were checked on, which limits how well
*any* calibrator can do.
""")

,chosen,Brier none,Brier sigmoid,Brier isotonic
model,,,,
Logistic regression,none,0.038,0.044,0.045
Random forest,none,0.042,0.043,0.043
LightGBM,sigmoid,0.435,0.043,0.045
EBM,sigmoid,0.051,0.044,0.047



**Inference.** The calibrator that helped most is for **LightGBM**
(Brier 0.435 uncalibrated -> 0.043).
That model's class-weighting option was **balanced**: its raw scores were on the wrong scale on purpose, so calibration rescued them.
Choices by model: Logistic regression: none, Random forest: none, LightGBM: sigmoid, EBM: sigmoid. Isotonic was
never selected: flexibility did not pay for
itself. Caveat from the data: the late rate was **11.4%** in the window the calibrators
were fitted on and **3.7%** in the window they were checked on, which limits how well
*any* calibrator can do.


## 7. The test set: one look
Everything above fixed the code and the primary model. The test period was then evaluated once, with
95% bootstrap intervals (orders resampled for PR-AUC; whole days for daily-capacity metrics).

In [7]:
tt = w.test_table()
fig, ax = plt.subplots(figsize=(9, 4))
y = np.arange(len(tt))[::-1]
ax.errorbar(
    tt["PR-AUC"],
    y,
    xerr=[tt["PR-AUC"] - tt["lo"], tt["hi"] - tt["PR-AUC"]],
    fmt="o",
    color="#222",
    capsize=3,
)
ax.set_yticks(y, tt["model"])
ax.axvline(res["population"]["late_rate"], color="grey", ls="--")
ax.text(
    res["population"]["late_rate"],
    y.max() + 0.5,
    " random ranking = base rate",
    color="grey",
    va="bottom",
)
ax.set_xlabel("PR-AUC on the test period (95% CI)")
ax.set_title("Test-set ranking quality")
plt.tight_layout()
plt.show()
display(tt.drop(columns=["key", "lo", "hi"]).set_index("model"))

,PR-AUC,ROC-AUC,Brier,lift@10%
model,,,,
Guess the training rate,0.045,0.500,0.044,1.028
Hand-written rule,0.048,0.527,0.044,0.938
Logistic regression,0.093,0.701,0.043,2.198
Random forest,0.075,0.665,0.046,1.626
LightGBM,0.078,0.683,0.043,1.957
EBM,0.074,0.656,0.043,1.644


In [8]:
pdiff = w.paired_differences()
display(pdiff.set_index("comparison"))
g = pdiff.set_index("comparison").loc["ebm minus lightgbm"]
verdict = (
    "includes zero: no reliable difference"
    if g["CI low"] <= 0 <= g["CI high"]
    else "excludes zero: a real difference"
)
best = tt.loc[tt["PR-AUC"].idxmax(), "model"]
Markdown(f"""
**Inference.**
- **The glass-box costs nothing:** EBM minus LightGBM is **{g["PR-AUC difference"]:+.3f}** PR-AUC,
  95% CI [{g["CI low"]:+.3f}, {g["CI high"]:+.3f}] - the interval {verdict}. If an interpretable
  boosted model is wanted, the EBM is a safe choice.
- The highest PR-AUC belongs to **{best}**, the simplest model: extra flexibility did not buy
  out-of-time accuracy in this data.
- Intervals are wide relative to the differences between the three non-linear models, so do not
  over-read the order among them.
""")

,PR-AUC difference,CI low,CI high
comparison,,,
ebm minus lightgbm,-0.004,-0.009,0.002
logreg minus lightgbm,0.015,0.009,0.024
ebm minus logreg,-0.019,-0.028,-0.012
logreg minus rule,0.045,0.039,0.055



**Inference.**
- **The glass-box costs nothing:** EBM minus LightGBM is **-0.004** PR-AUC,
  95% CI [-0.009, +0.002] - the interval includes zero: no reliable difference. If an interpretable
  boosted model is wanted, the EBM is a safe choice.
- The highest PR-AUC belongs to **Logistic regression**, the simplest model: extra flexibility did not buy
  out-of-time accuracy in this data.
- Intervals are wide relative to the differences between the three non-linear models, so do not
  over-read the order among them.


## 8. Does it hold up over time?
The test period is four months. Splitting it by month shows how ranking quality changes as time
passes after the training data ends.

In [9]:
mon = w.monthly()
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for m in ["logreg", "lightgbm", "ebm"]:
    g = mon[mon["model"] == m].sort_values("month")
    ax[0].plot(g["month"], g["roc_auc"], marker="o", label=w.LABEL[m])
    ax[1].plot(g["month"], g["lift_at_10"], marker="o", label=w.LABEL[m])
ax[0].axhline(0.5, color="grey", ls="--")
ax[0].set_title("ROC-AUC by month")
ax[0].legend()
ax[1].axhline(1.0, color="grey", ls="--")
ax[1].set_title("Lift at 10% daily capacity by month")
plt.tight_layout()
plt.show()
mp = mon[mon["model"] == prim].sort_values("month")
Markdown(f"""
**Inference.** For the primary model, ROC-AUC goes from **{mp.iloc[0]["roc_auc"]:.2f}** in
{mp.iloc[0]["month"]} to **{mp.iloc[-1]["roc_auc"]:.2f}** in {mp.iloc[-1]["month"]}, and lift at 10%
capacity from **{mp.iloc[0]["lift_at_10"]:.2f}** to **{mp.iloc[-1]["lift_at_10"]:.2f}**
(1.0 = no better than random). Ranking skill decays after training: in production this is handled
by monitoring and scheduled retraining, not by a cleverer static model.
""")


**Inference.** For the primary model, ROC-AUC goes from **0.75** in
2018-05 to **0.57** in 2018-08, and lift at 10%
capacity from **2.83** to **1.07**
(1.0 = no better than random). Ranking skill decays after training: in production this is handled
by monitoring and scheduled retraining, not by a cleverer static model.


## 9. What this walkthrough teaches (and what I would do next)

In [10]:
pm = res["models"][prim]["metrics"]["pr_auc"]["value"]
bm = res["population"]["late_rate"]
Markdown(f"""
**What survived the test**
- A one-line analyst rule did not; the primary model did (PR-AUC **{pm:.3f}** = **{pm / bm:.1f}x**
  the base rate of {bm:.3f}).
- The interpretable options are not a sacrifice (Section 7).
- Honest evaluation required *three* kinds of separation: tune vs calibrate vs test, all by time.
""")


**What survived the test**
- A one-line analyst rule did not; the primary model did (PR-AUC **0.093** = **2.0x**
  the base rate of 0.045).
- The interpretable options are not a sacrifice (Section 7).
- Honest evaluation required *three* kinds of separation: tune vs calibrate vs test, all by time.


**What I would do differently with more time**
1. Tune with expanding-window time-series CV so hyperparameters are chosen across several regimes,
   not one surge window.
2. Remove the calendar proxies (`approve_month`, holiday features) and re-validate.
3. Add drift monitoring and measure the intervention success rate in a pilot.

See `reports/model_card.md` for the single-page summary and `reports/results.md` for every table.